In [0]:
# ==========================================================
# 05_gold_alunos
# Camada: Gold
# Fonte: default.alunos (Silver)
# Destino: default.gold_alunos
#
# RESPONSABILIDADE
# Agregar a base de alunos por ano, município e rede,
# produzindo taxa de alfabetização e proficiência média
# ponderadas pelo peso amostral.
#
# NOTA DE ARQUITETURA
# peso_aluno é o peso amostral do INEP. Média simples
# distorce o resultado; a taxa precisa ser ponderada.
# presenca = 1 filtra apenas alunos que fizeram a prova.
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA_DESTINO = "default"
TABELA_DESTINO = "gold_alunos"

TABELA_ORIGEM = f"{CATALOGO}.{SCHEMA_DESTINO}.alunos"
TABELA_FULL = f"{CATALOGO}.{SCHEMA_DESTINO}.{TABELA_DESTINO}"

print(f"Origem : {TABELA_ORIGEM}")
print(f"Destino: {TABELA_FULL}")

In [0]:
df_raw = spark.table(TABELA_ORIGEM)

print(f"Registros lidos: {df_raw.count():,}")
print(f"Colunas: {len(df_raw.columns)}")
df_raw.printSchema()

In [0]:
df_gold = (
    df_raw
    .filter(F.col("presenca") == 1)
    .groupBy("ano", "id_municipio", "rede")
    .agg(
        F.count("*").alias("qtd_alunos"),
        F.countDistinct("id_escola").alias("qtd_escolas"),
        (F.sum(F.col("alfabetizado") * F.col("peso_aluno")) /
         F.sum("peso_aluno")).alias("taxa_alfabetizacao"),
        (F.sum(F.col("proficiencia") * F.col("peso_aluno")) /
         F.sum("peso_aluno")).alias("media_proficiencia"),
        F.sum("peso_aluno").alias("peso_total"),
    )
    .withColumn("taxa_alfabetizacao", F.round(F.col("taxa_alfabetizacao") * 100, 2))
    .withColumn("media_proficiencia", F.round(F.col("media_proficiencia"), 2))
)

print(f"Registros após agregação: {df_gold.count():,}")
df_gold.show(5, truncate=False)

In [0]:
df_gold = (
    df_gold
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source", F.lit("2026-11-01_alunos.parquet"))
    .withColumn("version", F.lit("v1"))
)

df_gold.select("ano", "id_municipio", "rede", "qtd_alunos",
               "taxa_alfabetizacao", "ingested_at", "source", "version").show(5, truncate=False)

In [0]:
total = df_gold.count()

nulos_taxa = df_gold.filter(F.col("taxa_alfabetizacao").isNull()).count()
nulos_rede = df_gold.filter(F.col("rede").isNull()).count()

fora_faixa = df_gold.filter(
    (F.col("taxa_alfabetizacao") < 0) | (F.col("taxa_alfabetizacao") > 100)
).count()

chave = ["ano", "id_municipio", "rede"]
duplicatas = total - df_gold.dropDuplicates(chave).count()

print("=" * 55)
print("DATA QUALITY — gold_alunos")
print("=" * 55)
print(f"Registros totais          : {total:,}")
print(f"Taxa nula                 : {nulos_taxa:,} ({nulos_taxa/total*100:.2f}%)")
print(f"Rede nula                 : {nulos_rede:,} ({nulos_rede/total*100:.2f}%)")
print(f"Taxa fora de 0-100        : {fora_faixa:,}")
print(f"Duplicatas na chave       : {duplicatas:,}")
print(f"Anos                      : {[r[0] for r in df_gold.select('ano').distinct().orderBy('ano').collect()]}")
print(f"Redes                     : {[r[0] for r in df_gold.select('rede').distinct().orderBy('rede').collect()]}")
print(f"Municípios distintos      : {df_gold.select('id_municipio').distinct().count():,}")
print("=" * 55)

assert nulos_taxa == 0, "FALHA CRÍTICA: taxa_alfabetizacao nula"
assert fora_faixa == 0, "FALHA CRÍTICA: taxa fora do domínio 0-100"
assert duplicatas == 0, "FALHA CRÍTICA: duplicatas na chave"
print("Todas as regras críticas passaram.")

In [0]:
(df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_FULL))

print(f"Gravado em {TABELA_FULL}")
print(f"Registros: {spark.table(TABELA_FULL).count():,}")

In [0]:
df_check = spark.table(TABELA_FULL)

print(f"Registros na tabela : {df_check.count():,}")
print(f"Colunas             : {len(df_check.columns)}")
print(f"Anos                : {[r[0] for r in df_check.select('ano').distinct().orderBy('ano').collect()]}")
print(f"Municípios          : {df_check.select('id_municipio').distinct().count():,}")

print("\nAmostra:")
df_check.select(
    "ano", "id_municipio", "rede", "qtd_alunos",
    "taxa_alfabetizacao", "media_proficiencia"
).show(5, truncate=False)

print("\nDistribuição por rede:")
df_check.groupBy("rede").count().orderBy("rede").show()

%md
## Nota de Arquitetura — Distribuição por Rede

A Gold `gold_alunos` apresenta a seguinte distribuição de registros por rede:

| Rede | Registros | Participação |
|------|-----------|--------------|
| 2    | 2.144     | 17,3%        |
| 3    | 10.272    | 82,7%        |
| 4    | 1         | 0,008%       |

**Por que a rede 4 tem apenas um registro?**

Não é falha de código nem de ingestão. A amostra do INEP concentra a avaliação
na rede pública, e a rede 4 aparece de forma residual em apenas um município.
O dado é legítimo, mas a representatividade estatística é desprezível.

**Implicação analítica**

A rede 4 não deve ser usada em comparações por rede. Qualquer média calculada
sobre ela reflete um único município e não pode ser generalizada. Nas análises
da camada Gold, recomenda-se filtrar `rede IN ('2','3')` ou agregar apenas
os níveis com volume mínimo de alunos.

**Decisão de modelagem**

O registro foi mantido na tabela por princípio de rastreabilidade — a Gold
reflete fielmente a Silver. A exclusão, quando necessária, é feita no consumo
(dashboard ou modelo), não na camada de dados.